In [1]:
!nvidia-smi
import torch
print("GPU count:", torch.cuda.device_count())

Wed Jul 15 21:25:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(hf_token)

In [3]:
%cd /kaggle/working

/kaggle/working


In [4]:
# Cài đặt các thư viện cần thiết (không cài đè PyTorch để tránh xung đột trên Kaggle)
!pip install -q transformers evaluate jiwer wandb pandas scikit-learn num2words

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.5 MB/s eta 0:00:00


# model_handling.py

In [5]:
%%writefile model_handling.py
from transformers import Wav2Vec2PreTrainedModel, Wav2Vec2Model
from torch import nn
import warnings
import torch
from transformers.modeling_outputs import CausalLMOutput
from collections import OrderedDict

_HIDDEN_STATES_START_POSITION = 2

class Wav2Vec2ForCTC(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.wav2vec2 = Wav2Vec2Model(config)
        self.dropout = nn.Dropout(config.final_dropout)
        self.feature_transform = nn.Sequential(OrderedDict([
            ('linear1', nn.Linear(config.hidden_size, config.hidden_size)),
            ('bn1', nn.BatchNorm1d(config.hidden_size)),
            ('activation1', nn.LeakyReLU()),
            ('drop1', nn.Dropout(config.final_dropout)),
            ('linear2', nn.Linear(config.hidden_size, config.hidden_size)),
            ('bn2', nn.BatchNorm1d(config.hidden_size)),
            ('activation2', nn.LeakyReLU()),
            ('drop2', nn.Dropout(config.final_dropout)),
            ('linear3', nn.Linear(config.hidden_size, config.hidden_size)),
            ('bn3', nn.BatchNorm1d(config.hidden_size)),
            ('activation3', nn.LeakyReLU()),
            ('drop3', nn.Dropout(config.final_dropout))
        ]))
        if config.vocab_size is None:
            raise ValueError(
                "You are trying to instantiate Wav2Vec2ForCTC with a configuration that "
                "does not define the vocabulary size of the language model head."
            )
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size)
        self.is_wav2vec_freeze = False
        self.post_init()

    def freeze_feature_extractor(self):
        self.freeze_feature_encoder()

    def freeze_feature_encoder(self):
        self.wav2vec2.feature_extractor._freeze_parameters()

    def freeze_wav2vec(self, is_freeze=True):
        if is_freeze:
            self.is_wav2vec_freeze = True
            for param in self.wav2vec2.parameters():
                param.requires_grad = False
        else:
            self.is_wav2vec_freeze = False
            for param in self.wav2vec2.parameters():
                param.requires_grad = True
        self.freeze_feature_encoder()
        model_total_params = sum(p.numel() for p in self.parameters())
        model_total_params_trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print("model_total_params: {}\nmodel_total_params_trainable: {}".format(model_total_params, model_total_params_trainable))

    def forward(
            self,
            input_values,
            attention_mask=None,
            output_attentions=None,
            output_hidden_states=None,
            return_dict=None,
            labels=None,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        outputs = self.wav2vec2(
            input_values,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        hidden_states = outputs[0]
        hidden_states = self.dropout(hidden_states)
        B, T, F = hidden_states.size()
        hidden_states = hidden_states.view(B * T, F)
        hidden_states = self.feature_transform(hidden_states)
        hidden_states = hidden_states.view(B, T, F)
        logits = self.lm_head(hidden_states)
        loss = None
        if labels is not None:
            if labels.max() >= self.config.vocab_size:
                raise ValueError(f"Label values must be <= vocab_size: {self.config.vocab_size}")
            attention_mask = (
                attention_mask if attention_mask is not None else torch.ones_like(input_values, dtype=torch.long)
            )
            input_lengths = self._get_feat_extract_output_lengths(attention_mask.sum(-1)).to(torch.long)
            labels_mask = labels >= 0
            target_lengths = labels_mask.sum(-1)
            flattened_targets = labels.masked_select(labels_mask)
            log_probs = nn.functional.log_softmax(logits, dim=-1, dtype=torch.float32).transpose(0, 1)
            with torch.backends.cudnn.flags(enabled=False):
                loss = nn.functional.ctc_loss(
                    log_probs,
                    flattened_targets,
                    input_lengths,
                    target_lengths,
                    blank=self.config.pad_token_id,
                    reduction=self.config.ctc_loss_reduction,
                    zero_infinity=self.config.ctc_zero_infinity,
                )
        if not return_dict:
            output = (logits,) + outputs[_HIDDEN_STATES_START_POSITION:]
            return ((loss,) + output) if loss is not None else output
        return CausalLMOutput(
            loss=loss, logits=logits, hidden_states=outputs.hidden_states, attentions=outputs.attentions
        )


Writing model_handling.py


# train.py

In [6]:
%%writefile train.py
import argparse
import json
import numpy as np
import os
import pandas as pd
import random
import re
import torch
import torchaudio
import wandb
import warnings
from dataclasses import dataclass
import evaluate
from pathlib import Path
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import (
    Wav2Vec2Processor,
    get_linear_schedule_with_warmup
)
from typing import List, Dict, Union, Optional, Any
from model_handling import Wav2Vec2ForCTC

warnings.filterwarnings("ignore")

class Config:
    # Paths
    data_dir: str = None
    train_csv: str = None
    val_csv: str = None
    test_csv: str = None
    output_dir: str = "./checkpoint_wav2vec2_vi"
    noise_dir: str = None

    # Model
    base_model: str = "nguyenvulebinh/wav2vec2-base-vi-vlsp2020"

    # Training
    batch_size: int = 16
    gradient_accumulation_steps: int = 2
    learning_rate: float = 5e-5
    weight_decay: float = 0.01
    num_epochs: int = 20
    warmup_ratio: float = 0.1

    # Optimization
    max_grad_norm: float = 1.0
    fp16: bool = True
    gradient_checkpointing: bool = True
    freeze_encoder_layers: int = 0

    # Data
    num_workers: int = 8
    prefetch_factor: int = 2

    # Augmentation settings
    augmentation_prob: float = 0.5
    noise_snr_range: tuple = (5, 25)
    gaussian_noise_level: float = 0.003
    time_shift_max: float = 0.1
    speed_range: tuple = (0.95, 1.05)
    volume_range: tuple = (0.8, 1.2)

    # Logging & Checkpoint intervals
    log_interval: int = 200
    save_interval: int = 200
    eval_steps: int = 200
    num_random_val_samples: int = 20
    num_best_worst_show: int = 3
    save_val_predictions: bool = True

    # Early stopping
    patience: int = 5
    min_delta: float = 0.001

    # Other
    seed: int = 42
    use_wandb: bool = False
    wandb_project: str = "wav2vec2-finetune-vi"

class AudioAugmentation:
    def __init__(self, config):
        self.config = config
        self.apply_prob = config.augmentation_prob
        self.noise_files = []
        self.noise_cache = {}
        if config.noise_dir and os.path.exists(config.noise_dir):
            self.load_noise_files(config.noise_dir)
            print(f"🔊 Loaded {len(self.noise_files)} noise files from {config.noise_dir}")
        else:
            print(f"⚠️ No noise directory provided or not found")

    def load_noise_files(self, noise_dir):
        noise_dir = Path(noise_dir)
        audio_extensions = {'.wav', '.mp3', '.flac', '.m4a', '.ogg', '.aac'}
        for ext in audio_extensions:
            self.noise_files.extend(list(noise_dir.glob(f'*{ext}')))
            self.noise_files.extend(list(noise_dir.glob(f'*{ext.upper()}')))
        self.noise_files = list(set(self.noise_files))

    def get_noise_segment(self, target_length, sample_rate=16000):
        if not self.noise_files:
            return None
        noise_file = random.choice(self.noise_files)
        try:
            cache_key = f"{noise_file}_{sample_rate}"
            if cache_key not in self.noise_cache:
                waveform, sr = torchaudio.load(str(noise_file))
                if sr != sample_rate:
                    resampler = torchaudio.transforms.Resample(sr, sample_rate)
                    waveform = resampler(waveform)
                if waveform.shape[0] > 1:
                    waveform = torch.mean(waveform, dim=0, keepdim=True)
                self.noise_cache[cache_key] = waveform.squeeze()
                if len(self.noise_cache) > 30:
                    oldest_key = next(iter(self.noise_cache))
                    del self.noise_cache[oldest_key]
            noise = self.noise_cache[cache_key]
            if len(noise) >= target_length:
                start_idx = random.randint(0, len(noise) - target_length)
                return noise[start_idx:start_idx + target_length]
            else:
                repeat_times = (target_length // len(noise)) + 1
                repeated_noise = noise.repeat(repeat_times)
                return repeated_noise[:target_length]
        except Exception as e:
            print(f"Error loading noise file {noise_file}: {e}")
            return None

    def add_real_noise(self, waveform):
        if np.random.random() > self.apply_prob or not self.noise_files:
            return waveform
        noise = self.get_noise_segment(len(waveform))
        if noise is None:
            return waveform
        signal_power = torch.mean(waveform ** 2)
        noise_power = torch.mean(noise ** 2)
        if noise_power == 0:
            return waveform
        target_snr_db = random.uniform(self.config.noise_snr_range[0], self.config.noise_snr_range[1])
        target_snr_linear = 10 ** (target_snr_db / 10)
        noise_scale = torch.sqrt(signal_power / (noise_power * target_snr_linear))
        return waveform + noise_scale * noise

    def add_gaussian_noise(self, waveform):
        if np.random.random() > self.apply_prob:
            return waveform
        noise = torch.randn_like(waveform) * self.config.gaussian_noise_level
        return waveform + noise

    def time_shift(self, waveform):
        if np.random.random() > self.apply_prob:
            return waveform
        shift = int(self.config.time_shift_max * waveform.shape[-1])
        shift_amount = np.random.randint(-shift, shift)
        return torch.roll(waveform, shift_amount, dims=-1)

    def speed_perturbation(self, waveform):
        if np.random.random() > self.apply_prob:
            return waveform
        speed_factor = random.uniform(self.config.speed_range[0], self.config.speed_range[1])
        if speed_factor != 1.0:
            original_length = len(waveform)
            new_length = int(original_length / speed_factor)
            indices = torch.linspace(0, original_length - 1, new_length)
            waveform_np = waveform.numpy()
            new_waveform = torch.tensor(
                np.interp(indices.numpy(), np.arange(original_length), waveform_np)
            ).float()
            if len(new_waveform) > original_length:
                return new_waveform[:original_length]
            elif len(new_waveform) < original_length:
                padding = original_length - len(new_waveform)
                return torch.nn.functional.pad(new_waveform, (0, padding))
            else:
                return new_waveform
        return waveform

    def volume_perturbation(self, waveform):
        if np.random.random() > self.apply_prob:
            return waveform
        volume_factor = random.uniform(self.config.volume_range[0], self.config.volume_range[1])
        return waveform * volume_factor

    def apply_augmentations(self, waveform):
        augmentations = [
            self.add_real_noise,
            self.add_gaussian_noise,
            self.time_shift,
            self.speed_perturbation,
            self.volume_perturbation
        ]
        random.shuffle(augmentations)
        augmented = waveform
        for aug_func in augmentations:
            augmented = aug_func(augmented)
        return augmented

class Wav2Vec2Dataset(Dataset):
    def __init__(self, df, processor, audio_dir, config=None, is_training=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.audio_dir = audio_dir
        self.is_training = is_training
        if is_training and config:
            self.augmentation = AudioAugmentation(config)
        else:
            self.augmentation = None
        self.valid_indices = []
        for idx, row in self.df.iterrows():
            audio_col = 'audio_path' if 'audio_path' in row else 'file_path'
            if audio_col not in row:
                audio_col = 'path'
            audio_path = os.path.join(self.audio_dir, row[audio_col])
            if os.path.exists(audio_path):
                self.valid_indices.append(idx)

    def get_audio_path(self, idx):
        real_idx = self.valid_indices[idx]
        row = self.df.iloc[real_idx]
        audio_col = 'audio_path' if 'audio_path' in row else ('file_path' if 'file_path' in row else 'path')
        return row[audio_col]

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        real_idx = self.valid_indices[idx]
        row = self.df.iloc[real_idx]
        audio_col = 'audio_path' if 'audio_path' in row else ('file_path' if 'file_path' in row else 'path')
        text_col = 'transcription' if 'transcription' in row else 'text'
        audio_path = os.path.join(self.audio_dir, row[audio_col])
        try:
            waveform, sr = torchaudio.load(audio_path)
            if sr != 16000:
                waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)
            waveform = waveform.squeeze()
            if self.is_training and self.augmentation:
                waveform = self.augmentation.apply_augmentations(waveform)
            input_values = self.processor(
                waveform.numpy(),
                sampling_rate=16000,
                return_tensors="pt"
            ).input_values[0]
            text = clean_text(row[text_col])
            labels = self.processor.tokenizer(text).input_ids
            return {
                "input_values": input_values,
                "labels": labels,
                "text": text
            }
        except Exception as e:
            print(f"Error processing {audio_path}: {e}")
            return self.__getitem__((idx + 1) % len(self))

@dataclass
class CTCDataCollator:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_attention_mask=True,
            return_tensors="pt"
        )
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def clean_text(text: str) -> str:
    text = text.lower()
    
    # 1. Xử lý khoảng số (Ví dụ: 11-40 -> 11 đến 40, 2-3 ngày -> 2 đến 3 ngày)
    text = re.sub(r'(\d+)\s*-\s*(\d+)', r'\1 đến \2', text)
    
    # 2. Bảo vệ số thập phân dạng tiếng Việt (Ví dụ: 2,2 -> 2 phẩy 2)
    text = re.sub(r'(\d+),(\d+)', r'\1 phẩy \2', text)
    
    # 3. Ánh xạ ký hiệu toán học & ký tự đặc biệt thành chữ nói
    sign_map = {
        "+": " cộng ",
        "→": " cho ra ", "->": " cho ra ",
        "=": " bằng ",
        "≥": " lớn hơn hoặc bằng ",
        "≤": " nhỏ hơn hoặc bằng ",
        "<": " nhỏ hơn ",
        ">": " lớn hơn ",
        "~": " khoảng ",
        "±": " cộng trừ ",
        "×": " nhân ", "·": " nhân ",
        "*": " sao ",
        "/": " trên ",
        "%": " phần trăm ",
        "α": " alpha ", "beta": " bê ta ", "β": " bê ta ", "gamma": " ga ma "
    }
    for sign, replacement in sign_map.items():
        text = text.replace(sign, replacement)
        
    # 4. Tách các đơn vị phổ biến dính liền với số (Ví dụ: 200mg -> 200 mg)
    text = re.sub(r'(\d+)\s*(mg|ml|l|g|kg|m2|m²|mmol|meq|micromol)\b', r'\1 \2', text)
    
    # 5. Từ điển dịch các chữ cái viết tắt / đơn vị / ký tự Latinh sang âm tiếng Việt
    word_speech_map = {
        "mg": "mi li gam", "ml": "mi li lít", "g": "gam", "kg": "ki lô gam",
        "mmol": "mi li mol", "meq": "mi li đương lượng", "m²": "mét vuông", "m2": "mét vuông",
        "cyp": "xê y pê", "ugt": "u gi tê", "g6pd": "gê sáu pê đê", "cd": "xê đê",
        "rna": "ác en nờ", "dna": "đê en nờ", "hcv": "hắc cê vê", "hiv": "hắc i vê",
        "tnf": "tê en ef", "il": "i lờ", "ns": "en ét", "ld": "el đê", "mic": "em i xê",
        "auc": "a u xê", "clcr": "xê lờ xê e rờ", "crcl": "xê e rờ xê lờ", "fda": "ef đê a",
        "a": "a", "b": "b", "c": "c", "d": "đê", "e": "e", "f": "ef", "g": "gê", 
        "h": "hắc", "i": "i", "k": "ca", "l": "lờ", "m": "em", "n": "en", 
        "o": "ô", "p": "pê", "r": "e rờ", "s": "ét", "t": "tê", "u": "u", "v": "vê", "x": "xê", "y": "y"
    }

    # Thư viện chuyển số thành chữ (Bắt buộc dùng bản vi để đọc tiếng Việt chuẩn)
    # Nếu chưa có hãy cài: pip install num2words
    from num2words import num2words
    
    processed_words = []
    for word in text.split():
        # Xóa các dấu câu bao quanh từ (nhưng giữ lại chữ và số bên trong)
        clean_w = re.sub(r'^[\,\?\.\!\;\:\'\"\(\)\[\]\{\}\<\>]+|[\,\?\.\!\;\:\'\"\(\)\[\]\{\}\<\>]+$', '', word)
        
        if not clean_w:
            continue
            
        # Nếu từ nằm trong từ điển dịch giọng đọc (ví dụ: 'mg', 'cyp')
        if clean_w in word_speech_map:
            processed_words.append(word_speech_map[clean_w])
        # Nếu là số nguyên thuần túy
        elif clean_w.isdigit():
            processed_words.append(num2words(int(clean_w), lang='vi'))
        # Nếu là dạng text kết hợp số (Ví dụ: cyp2d6, cahpo4)
        elif re.search(r'[a-z]+.*\d+|\d+.*[a-z]+', clean_w):
            # Tách chuỗi thành các phần chữ và số riêng biệt bằng regex
            parts = re.findall(r'[a-zăâđêôơư]+|\d+', clean_w)
            sub_processed = []
            for part in parts:
                if part.isdigit():
                    sub_processed.append(num2words(int(part), lang='vi'))
                elif part in word_speech_map:
                    sub_processed.append(word_speech_map[part])
                else:
                    # Đánh vần từng ký tự Latin lẻ nếu là tên công thức hóa học/receptor phức tạp
                    spelled = [word_speech_map.get(char, char) for char in part]
                    sub_processed.append(" ".join(spelled))
            processed_words.append(" ".join(sub_processed))
        else:
            # Giữ nguyên các từ tiếng Việt bình thường
            processed_words.append(clean_w)
            
    # Kết hợp lại và dọn dẹp khoảng trắng
    final_text = " ".join(processed_words)
    final_text = re.sub(r'\s+', ' ', final_text)
    return final_text.strip()

def load_data(config):
    # Load train/val data từ file riêng biệt
    train_df = pd.read_csv(config.train_csv)
    val_df = pd.read_csv(config.val_csv)
    print(f"📊 Dataset statistics:")
    print(f"   Train samples: {len(train_df)}")
    print(f"   Val samples: {len(val_df)}")
    return train_df, val_df

def setup_model(config):
    processor = Wav2Vec2Processor.from_pretrained(config.base_model, additional_special_tokens=[])
    model = Wav2Vec2ForCTC.from_pretrained(
        config.base_model,
        pad_token_id=processor.tokenizer.pad_token_id,
        vocab_size=len(processor.tokenizer)
    )
    model.freeze_feature_encoder()
    if config.freeze_encoder_layers > 0:
        for i, layer in enumerate(model.wav2vec2.encoder.layers):
            if i < config.freeze_encoder_layers:
                for param in layer.parameters():
                    param.requires_grad = False
        print(f"❄️ Froze first {config.freeze_encoder_layers} encoder layers")
    if config.gradient_checkpointing:
        model.gradient_checkpointing_enable()
        print("✅ Gradient checkpointing enabled")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Using device: {device}")
    model.to(device)
    if torch.cuda.device_count() > 1:
        print(f"🔥 Using {torch.cuda.device_count()} GPUs")
        model = torch.nn.DataParallel(model)
    return model, processor, device

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_score):
        if self.best_score is None:
            self.best_score = val_score
        elif val_score > self.best_score - self.min_delta:
            self.counter += 1
            print(f"⚠️ EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_score
            self.counter = 0

def validate(model, val_loader, processor, device, config, val_dataset=None):
    model.eval()
    total_loss = 0
    total_steps = 0
    all_preds = []
    all_refs = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating", leave=False):
            input_values = batch["input_values"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            with autocast(enabled=config.fp16):
                outputs = model(input_values=input_values, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
            if torch.cuda.device_count() > 1:
                loss = loss.mean()
            total_loss += loss.item()
            total_steps += 1
            pred_ids = torch.argmax(outputs.logits, dim=-1)
            pred_strs = processor.batch_decode(pred_ids)
            label_ids = labels.clone()
            label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
            label_strs = processor.batch_decode(label_ids, group_tokens=False)
            all_preds.extend(pred_strs)
            all_refs.extend(label_strs)
    avg_loss = total_loss / total_steps
    wer_metric = evaluate.load("wer")
    wer = wer_metric.compute(predictions=all_preds, references=all_refs)
    random_predictions = []
    if val_dataset is not None and len(val_dataset) > 0 and config.num_random_val_samples > 0:
        num_samples = min(config.num_random_val_samples, len(val_dataset))
        random_indices = np.random.choice(len(val_dataset), num_samples, replace=False)
        for idx in tqdm(random_indices, desc="Random predictions", leave=False):
            try:
                sample = val_dataset[idx]
                input_values = sample["input_values"].unsqueeze(0).to(device)
                with autocast(enabled=config.fp16):
                    outputs = model(input_values=input_values)
                    pred_ids = torch.argmax(outputs.logits, dim=-1)
                pred_str = processor.batch_decode(pred_ids)[0]
                ref_str = sample["text"]
                sample_wer = wer_metric.compute(predictions=[pred_str], references=[ref_str])
                audio_path = 'N/A'
                if hasattr(val_dataset, 'get_audio_path'):
                    try:
                        audio_path = val_dataset.get_audio_path(idx)
                    except:
                        pass
                random_predictions.append({
                    'reference': ref_str,
                    'prediction': pred_str,
                    'wer': sample_wer,
                    'index': int(idx),
                    'audio_path': audio_path
                })
            except Exception as e:
                print(f"Error processing sample {idx}: {e}")
                continue
    return avg_loss, wer, all_preds, all_refs, random_predictions

def save_checkpoint(model, optimizer, scheduler, epoch, step, wer, config, processor, is_best=False):
    os.makedirs(config.output_dir, exist_ok=True)
    model_to_save = model.module if hasattr(model, 'module') else model
    checkpoint = {
        'epoch': epoch,
        'step': step,
        'wer': wer,
        'model_state': model_to_save.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'config': config.__dict__
    }
    if is_best:
        path = os.path.join(config.output_dir, 'best_model.pt')
        model_to_save.save_pretrained(os.path.join(config.output_dir, 'best_model_hf'))
        processor.save_pretrained(os.path.join(config.output_dir, 'best_model_hf'))
    else:
        path = os.path.join(config.output_dir, f'checkpoint_step_{step}.pt')
    torch.save(checkpoint, path)
    processor_path = os.path.join(config.output_dir, 'processor')
    if not os.path.exists(processor_path):
        processor.save_pretrained(processor_path)
    print(f"💾 Saved checkpoint: {path}")

def find_last_checkpoint(output_dir: str) -> Optional[str]:
    if not os.path.isdir(output_dir):
        return None
    checkpoints = []
    for f in os.listdir(output_dir):
        if f.startswith('checkpoint_step_') and f.endswith('.pt'):
            try:
                step = int(f.replace('checkpoint_step_', '').replace('.pt', ''))
                checkpoints.append((step, os.path.join(output_dir, f)))
            except:
                continue
    if not checkpoints:
        return None
    checkpoints.sort(key=lambda x: x[0])
    return checkpoints[-1][1]

def load_checkpoint(path, model, optimizer=None, scheduler=None):
    checkpoint = torch.load(path)
    model_to_load = model.module if hasattr(model, 'module') else model
    model_to_load.load_state_dict(checkpoint['model_state'])
    if optimizer:
        optimizer.load_state_dict(checkpoint['optimizer_state'])
    if scheduler:
        scheduler.load_state_dict(checkpoint['scheduler_state'])
    return checkpoint['epoch'], checkpoint['step'], checkpoint.get('wer', float('inf'))

def train(args):
    config = Config()
    for key, value in vars(args).items():
        if hasattr(config, key):
            setattr(config, key, value)
    set_seed(config.seed)
    if config.use_wandb:
        wandb.init(
            project=config.wandb_project,
            config=config.__dict__,
            name=f"wav2vec2-vi-{config.base_model.split('/')[-1]}"
        )
    train_df, val_df = load_data(config)
    model, processor, device = setup_model(config)
    train_dataset = Wav2Vec2Dataset(
        df=train_df,
        processor=processor,
        audio_dir=config.data_dir,
        config=config,
        is_training=True
    )
    val_dataset = Wav2Vec2Dataset(
        df=val_df,
        processor=processor,
        audio_dir=config.data_dir,
        config=None,
        is_training=False
    )
    data_collator = CTCDataCollator(processor=processor)
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        collate_fn=data_collator,
        num_workers=config.num_workers,
        pin_memory=True,
        persistent_workers=True if config.num_workers > 0 else False,
        prefetch_factor=config.prefetch_factor
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size * 2,
        shuffle=False,
        collate_fn=data_collator,
        num_workers=config.num_workers,
        pin_memory=True,
        persistent_workers=True if config.num_workers > 0 else False
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )
    num_training_steps = len(train_loader) * config.num_epochs // config.gradient_accumulation_steps
    num_warmup_steps = int(config.warmup_ratio * num_training_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )
    scaler = GradScaler(enabled=config.fp16)
    early_stopping = EarlyStopping(patience=config.patience, min_delta=config.min_delta)
    start_epoch = 0
    global_step = 0
    best_wer = float('inf')
    if args.resume or args.resume_from_checkpoint:
        if args.resume_from_checkpoint:
            resume_path = args.resume_from_checkpoint
        else:
            resume_path = find_last_checkpoint(config.output_dir)
        if resume_path and os.path.exists(resume_path):
            print(f"🔄 Resuming from checkpoint: {resume_path}")
            start_epoch, global_step, best_wer = load_checkpoint(
                resume_path, model, optimizer, scheduler
            )
            print(f"   Resumed from epoch {start_epoch}, step {global_step}, best WER {best_wer:.4f}")
        else:
            print("⚠️ No checkpoint found, starting from scratch")

    PHASE_1_EPOCHS = 5
    
    print("\n🏃 Starting training...")
    print(f"📊 Total training steps: {num_training_steps}")
    print(f"🔥 Warmup steps: {num_warmup_steps}")
    print(f"📦 Effective batch size: {config.batch_size * config.gradient_accumulation_steps}")
    for epoch in range(start_epoch, config.num_epochs):
        print(f"\n{'=' * 50}")
        print(f"📅 Epoch {epoch + 1}/{config.num_epochs}")
        print(f"{'=' * 50}")
        
        # ---------------------------------------------------------
        # 🧠 LOGIC 2-STAGE TRAINING 
        # ---------------------------------------------------------
        model_to_train = model.module if hasattr(model, 'module') else model
        
        if epoch < PHASE_1_EPOCHS:
            # Giai đoạn 1: Đóng băng toàn bộ trừ lớp Classification Head
            if epoch == start_epoch:
                print("🔥 GIAI ĐOẠN 1: Đóng băng toàn bộ Transformer, chỉ huấn luyện LM Head...")
                for param in model_to_train.parameters():
                    param.requires_grad = False
                for param in model_to_train.lm_head.parameters():
                    param.requires_grad = True
                    
        elif epoch == PHASE_1_EPOCHS:
            # Chuyển giao sang Giai đoạn 2
            print(f"🔥 GIAI ĐOẠN 2: Mở khóa Transformer (Đóng băng {config.freeze_encoder_layers} layers đầu)...")
            for param in model_to_train.parameters():
                param.requires_grad = True
            
            model_to_train.freeze_feature_encoder()
            
            if config.freeze_encoder_layers > 0:
                for i, layer in enumerate(model_to_train.wav2vec2.encoder.layers):
                    if i < config.freeze_encoder_layers:
                        for param in layer.parameters():
                            param.requires_grad = False
            
            # Can thiệp an toàn vào scheduler để hạ LR xuống 5 lần
            for i in range(len(scheduler.base_lrs)):
                scheduler.base_lrs[i] = scheduler.base_lrs[i] / 5.0
            print("📉 Đã giảm Base Learning Rate xuống 5 lần để tránh Catastrophic Forgetting!")
            
        elif epoch > PHASE_1_EPOCHS and epoch == start_epoch:
            # Xử lý trường hợp resume training giữa chừng ở Giai đoạn 2
            print(f"🔄 Tiếp tục GIAI ĐOẠN 2 (Đã đóng băng {config.freeze_encoder_layers} layers đầu)...")
            for param in model_to_train.parameters():
                param.requires_grad = True
            
            model_to_train.freeze_feature_encoder()
            
            if config.freeze_encoder_layers > 0:
                for i, layer in enumerate(model_to_train.wav2vec2.encoder.layers):
                    if i < config.freeze_encoder_layers:
                        for param in layer.parameters():
                            param.requires_grad = False
        # ---------------------------------------------------------

        model.train()
        epoch_loss = 0
        epoch_steps = 0
        pbar = tqdm(train_loader, desc=f"Training Epoch {epoch + 1}")
        
        for step, batch in enumerate(pbar):
            input_values = batch["input_values"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            with autocast(enabled=config.fp16):
                outputs = model(input_values=input_values, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                if torch.cuda.device_count() > 1:
                    loss = loss.mean()
                loss = loss / config.gradient_accumulation_steps
            scaler.scale(loss).backward()
            if (step + 1) % config.gradient_accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1
                current_lr = scheduler.get_last_lr()[0]
                pbar.set_postfix({
                    'loss': f'{loss.item() * config.gradient_accumulation_steps:.4f}',
                    'lr': f'{current_lr:.2e}'
                })
                if config.use_wandb:
                    wandb.log({
                        'train_loss': loss.item() * config.gradient_accumulation_steps,
                        'learning_rate': current_lr,
                        'epoch': epoch,
                        'global_step': global_step
                    })
                epoch_loss += loss.item() * config.gradient_accumulation_steps
                epoch_steps += 1
                if global_step % config.log_interval == 0:
                    print(f"\n🔍 Running validation at step {global_step}...")
                    val_loss, val_wer, preds, refs, random_preds = validate(
                        model, val_loader, processor, device, config, val_dataset
                    )
                    print(f"📊 Validation Results:")
                    print(f"   Loss: {val_loss:.4f}")
                    print(f"   WER: {val_wer:.4f}")
                    if config.use_wandb:
                        wandb.log({
                            'val_loss': val_loss,
                            'val_wer': val_wer,
                            'global_step': global_step
                        })
                    if random_preds:
                        print(f"\n📝 Random predictions ({len(random_preds)} samples):")
                        print("=" * 80)
                        random_preds_sorted = sorted(random_preds, key=lambda x: x['wer'])
                        num_show = min(config.num_best_worst_show, len(random_preds) // 2)
                        if num_show > 0:
                            print(f"\n🎯 BEST {num_show} predictions (lowest WER):")
                            for i, pred_dict in enumerate(random_preds_sorted[:num_show], 1):
                                print(f"\n[Best {i}] Index: {pred_dict['index']} | WER: {pred_dict['wer']:.3f}")
                                if pred_dict.get('audio_path', 'N/A') != 'N/A':
                                    print(f"FILE: {pred_dict['audio_path']}")
                                print(f"REF:  {pred_dict['reference']}")
                                print(f"PRED: {pred_dict['prediction']}")
                            print(f"\n\n❌ WORST {num_show} predictions (highest WER):")
                            for i, pred_dict in enumerate(random_preds_sorted[-num_show:], 1):
                                print(f"\n[Worst {i}] Index: {pred_dict['index']} | WER: {pred_dict['wer']:.3f}")
                                if pred_dict.get('audio_path', 'N/A') != 'N/A':
                                    print(f"FILE: {pred_dict['audio_path']}")
                                print(f"REF:  {pred_dict['reference']}")
                                print(f"PRED: {pred_dict['prediction']}")
                        if len(random_preds) <= 10:
                            print("\n\n📋 ALL predictions:")
                            for i, pred_dict in enumerate(random_preds_sorted, 1):
                                print(f"\n[{i}] Index: {pred_dict['index']} | WER: {pred_dict['wer']:.3f}")
                                if pred_dict.get('audio_path', 'N/A') != 'N/A':
                                    print(f"FILE: {pred_dict['audio_path']}")
                                print(f"REF:  {pred_dict['reference']}")
                                print(f"PRED: {pred_dict['prediction']}")
                        wers = [p['wer'] for p in random_preds]
                        print(f"\n\n📊 Random samples statistics:")
                        print(f"   Average WER: {np.mean(wers):.3f}")
                        print(f"   Median WER:  {np.median(wers):.3f}")
                        print(f"   Min WER:     {np.min(wers):.3f}")
                        print(f"   Max WER:     {np.max(wers):.3f}")
                        print(f"   Std WER:     {np.std(wers):.3f}")
                        print("=" * 80)
                        if config.save_val_predictions:
                            pred_file = os.path.join(
                                config.output_dir,
                                f'val_predictions_step_{global_step}.json'
                            )
                            os.makedirs(config.output_dir, exist_ok=True)
                            with open(pred_file, 'w', encoding='utf-8') as f:
                                json.dump({
                                    'step': global_step,
                                    'epoch': epoch + 1,
                                    'overall_wer': float(val_wer),
                                    'random_samples_stats': {
                                        'mean_wer': float(np.mean(wers)),
                                        'median_wer': float(np.median(wers)),
                                        'min_wer': float(np.min(wers)),
                                        'max_wer': float(np.max(wers)),
                                        'std_wer': float(np.std(wers))
                                    },
                                    'predictions': random_preds_sorted
                                }, f, ensure_ascii=False, indent=2)
                            print(f"💾 Saved predictions to {pred_file}")
                    if val_wer < best_wer:
                        best_wer = val_wer
                        save_checkpoint(
                            model, optimizer, scheduler, epoch, global_step, val_wer,
                            config, processor, is_best=True
                        )
                        print(f"🎉 New best WER: {best_wer:.4f}")
                    if global_step % config.save_interval == 0:
                        save_checkpoint(
                            model, optimizer, scheduler, epoch, global_step, val_wer,
                            config, processor, is_best=False
                        )
                    early_stopping(val_wer)
                    if early_stopping.early_stop:
                        print("🛑 Early stopping triggered!")
                        break
                    model.train()
                if global_step % 100 == 0:
                    torch.cuda.empty_cache()
        avg_epoch_loss = epoch_loss / epoch_steps if epoch_steps > 0 else 0
        print(f"\n📊 Epoch {epoch + 1} Summary:")
        print(f"   Average Loss: {avg_epoch_loss:.4f}")
        print(f"   Best WER: {best_wer:.4f}")
        if early_stopping.early_stop:
            break
    print(f"\n🎊 Training Complete!")
    print(f"📈 Best WER: {best_wer:.4f}")
    print(f"💾 Best model saved at: {os.path.join(config.output_dir, 'best_model.pt')}")
    model_to_save = model.module if hasattr(model, 'module') else model
    model_to_save.save_pretrained(config.output_dir)
    processor.save_pretrained(config.output_dir)
    if config.use_wandb:
        wandb.finish()

def evaluate_model(args):
    # Chạy đánh giá riêng biệt trên Test set
    config = Config()
    for key, value in vars(args).items():
        if hasattr(config, key):
            setattr(config, key, value)
    set_seed(config.seed)
    if not config.test_csv:
        raise ValueError("Phải cung cấp --test_csv để thực hiện đánh giá")
    print(f"Loading test data from {config.test_csv}...")
    test_df = pd.read_csv(config.test_csv)
    model, processor, device = setup_model(config)
    
    # Load weights
    if args.resume_from_checkpoint:
        checkpoint_path = args.resume_from_checkpoint
    else:
        checkpoint_path = os.path.join(config.output_dir, 'best_model.pt')
        
    if os.path.exists(checkpoint_path):
        print(f"🔄 Loading weights from {checkpoint_path}...")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model_to_load = model.module if hasattr(model, 'module') else model
        model_to_load.load_state_dict(checkpoint['model_state'])
        print("✅ Model weights loaded successfully.")
    else:
        print(f"⚠️ Checkpoint not found at {checkpoint_path}. Evaluating using base weights.")

    test_dataset = Wav2Vec2Dataset(
        df=test_df,
        processor=processor,
        audio_dir=config.data_dir,
        config=None,
        is_training=False
    )
    data_collator = CTCDataCollator(processor=processor)
    test_loader = DataLoader(
        test_dataset,
        batch_size=config.batch_size * 2,
        shuffle=False,
        collate_fn=data_collator,
        num_workers=config.num_workers,
        pin_memory=True
    )
    print(f"Evaluating {len(test_dataset)} samples...")
    avg_loss, test_wer, all_preds, all_refs, random_preds = validate(
        model, test_loader, processor, device, config, test_dataset
    )
    print("\n" + "="*50)
    print("📊 FINAL TEST SET RESULTS:")
    print(f"   Test Loss: {avg_loss:.4f}")
    print(f"   Test WER:  {test_wer:.4f}")
    print("="*50)
    
    pred_file = os.path.join(config.output_dir, 'test_predictions.json')
    os.makedirs(config.output_dir, exist_ok=True)
    with open(pred_file, 'w', encoding='utf-8') as f:
        json.dump({
            'overall_wer': float(test_wer),
            'predictions': random_preds
        }, f, ensure_ascii=False, indent=2)
    print(f"💾 Saved test predictions to {pred_file}")

def get_args():
    parser = argparse.ArgumentParser("Fine-tune Wav2Vec2 Vietnamese with Augmentation")
    # Data
    parser.add_argument("--data_dir", required=True, help="Directory containing audio files")
    parser.add_argument("--train_csv", required=True, help="CSV containing train split metadata")
    parser.add_argument("--val_csv", required=True, help="CSV containing val split metadata")
    parser.add_argument("--test_csv", help="CSV containing test split metadata")
    parser.add_argument("--noise_dir", help="Directory containing noise files for augmentation")
    parser.add_argument("--metadata_csv", help="Legacy argument")
    
    # Output
    parser.add_argument("--output_dir", default="./checkpoint_wav2vec2_vi")
    # Model
    parser.add_argument("--base_model", default="nguyenvulebinh/wav2vec2-base-vi-vlsp2020")
    # Training
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--num_epochs", type=int, default=20)
    parser.add_argument("--learning_rate", type=float, default=5e-5)
    parser.add_argument("--val_split", type=float, default=0.01)
    
    # Intervals
    parser.add_argument("--log_interval", type=int, default=200)
    parser.add_argument("--save_interval", type=int, default=200)
    parser.add_argument("--eval_steps", type=int, default=200)
    
    # Validation
    parser.add_argument("--num_random_val_samples", type=int, default=20)
    parser.add_argument("--save_val_predictions", action="store_true")
    # Augmentation
    parser.add_argument("--augmentation_prob", type=float, default=0.9)
    parser.add_argument("--no_augmentation", action="store_true")
    # Modes
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--resume_from_checkpoint", type=str)
    parser.add_argument("--eval_only", action="store_true")
    # Other
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--use_wandb", action="store_true")
    # Layers
    parser.add_argument("--freeze_encoder_layers", type=int, default=0)
    return parser.parse_args()

if __name__ == "__main__":
    args = get_args()
    if args.no_augmentation:
        args.augmentation_prob = 0.0
    if args.eval_only:
        evaluate_model(args)
    else:
        train(args)

Writing train.py


### 🔊 Hướng dẫn cấu trúc thư mục Tiếng ồn (Noise Directory)
Để sử dụng tính năng **Audio Augmentation (Real Noise)** trong quá trình huấn luyện, bạn cần chuẩn bị thư mục tiếng ồn như sau:

#### 1. Các Dataset tiếng ồn khuyên dùng:
- **MUSAN (Noise portion)**: Đây là tập hợp phổ biến nhất cho việc huấn luyện nhận dạng giọng nói, chứa khoảng 6 giờ âm thanh tiếng ồn (tiếng văn phòng, tiếng đồ vật, tiếng động cơ, tiếng môi trường...). Bạn có thể tìm thấy bản public trên Kaggle bằng cách tìm kiếm `"musan"` hoặc `"musan dataset"`.
- **DEMAND Dataset**: Chứa tiếng ồn nền chất lượng cao ghi ở các môi trường thực tế (quán cafe, văn phòng, công viên, đường phố...).
- **Custom Noise**: Bạn có thể tự tải lên các file âm thanh tiếng ồn mong muốn của riêng mình dưới dạng `.wav`, `.mp3` để làm tiếng ồn nền tăng cường.

#### 2. Cách tích hợp vào Kaggle:
1. Nhấp chọn **"Add Input"** trên giao diện Notebook Kaggle.
2. Tìm kiếm và thêm dataset tiếng ồn (ví dụ: `musan` hoặc một dataset tiếng ồn tùy chỉnh bạn đã tạo).
3. Copy đường dẫn thư mục tiếng ồn (ví dụ: `/kaggle/input/musan-dataset/noise/` hoặc `/kaggle/input/demand-noise-dataset/`) và điền vào biến `NOISE_DIR` bên dưới.


In [7]:
import os

# Thư mục dữ liệu chứa file audio và các file split metadata CSV
# Thay đổi đường dẫn này theo Kaggle của bạn (ví dụ: /kaggle/input/your-dataset-name/train_set)
DATA_DIR = "/kaggle/input/datasets/tracvanngocphuc/medicaldialog/train_set"

TRAIN_CSV = os.path.join(DATA_DIR, "train_split.csv")
VAL_CSV = os.path.join(DATA_DIR, "val_split.csv")
TEST_CSV = os.path.join(DATA_DIR, "test_split.csv")

# Thư mục chứa các file tiếng ồn phục vụ Augmentation (Để None nếu không sử dụng)
NOISE_DIR = "/kaggle/input/datasets/nhattruongdev/musan-noise/musan/noise/free-sound"

# Kiểm tra sự tồn tại của các file dữ liệu
for name, path in [("Train CSV", TRAIN_CSV), ("Val CSV", VAL_CSV), ("Test CSV", TEST_CSV)]:
    if os.path.exists(path):
        print(f"✅ Tìm thấy {name}: {path}")
    else:
        print(f"❌ KHÔNG tìm thấy {name}: {path}")

✅ Tìm thấy Train CSV: /kaggle/input/datasets/tracvanngocphuc/medicaldialog/train_set/train_split.csv
✅ Tìm thấy Val CSV: /kaggle/input/datasets/tracvanngocphuc/medicaldialog/train_set/val_split.csv
✅ Tìm thấy Test CSV: /kaggle/input/datasets/tracvanngocphuc/medicaldialog/train_set/test_split.csv


In [8]:
!python train.py \
    --data_dir "{DATA_DIR}" \
    --train_csv "{TRAIN_CSV}" \
    --val_csv "{VAL_CSV}" \
    --noise_dir "{NOISE_DIR}" \
    --output_dir /kaggle/working/checkpoints/wav2vec2_vi \
    --base_model nguyenvulebinh/wav2vec2-base-vi-vlsp2020 \
    --batch_size 16 \
    --num_epochs 30 \
    --learning_rate 3e-5 \
    --log_interval 200 \
    --save_interval 200 \
    --num_random_val_samples 20 \
    --save_val_predictions \
    --augmentation_prob 0.3 \
    --freeze_encoder_layers 8

📊 Dataset statistics:
   Train samples: 2727
   Val samples: 120
preprocessor_config.json: 100%|████████████████| 263/263 [00:00<00:00, 1.50MB/s]
config.json: 2.07kB [00:00, 1.03MB/s]
tokenizer_config.json: 100%|███████████████████| 396/396 [00:00<00:00, 2.79MB/s]
vocab.json: 1.17kB [00:00, 3.64MB/s]
added_tokens.json: 100%|██████████████████████| 30.0/30.0 [00:00<00:00, 217kB/s]
special_tokens_map.json: 2.59kB [00:00, 7.69MB/s]
pytorch_model.bin: 100%|█████████████████████| 385M/385M [00:04<00:00, 91.1MB/s]
model.safetensors:   0%|                             | 0.00/385M [00:00<?, ?B/s]
Loading weights:   0%|                                  | 0/234 [00:00<?, ?it/s]
Loading weights:   0%| | 1/234 [00:00<00:00, 1802.45it/s, Materializing param=fe
Loading weights:   0%| | 1/234 [00:00<00:00, 565.35it/s, Materializing param=fea
Loading weights:   1%| | 2/234 [00:00<00:00, 516.83it/s, Materializing param=fea
Loading weights:   1%| | 2/234 [00:00<00:00, 476.92it/s, Materializing param=fea


In [9]:
# Chạy đánh giá trên tập Test sau khi đã hoàn thành fine-tuning
!python train.py --eval_only \
    --data_dir "{DATA_DIR}" \
    --train_csv "{TRAIN_CSV}" \
    --val_csv "{VAL_CSV}" \
    --test_csv "{TEST_CSV}" \
    --output_dir /kaggle/working/checkpoints/wav2vec2_vi \
    --batch_size 16

Loading test data from /kaggle/input/datasets/tracvanngocphuc/medicaldialog/train_set/test_split.csv...
Loading weights: 100%|█| 234/234 [00:00<00:00, 2203.98it/s, Materializing param=
✅ Gradient checkpointing enabled
🚀 Using device: cuda
🔥 Using 2 GPUs
🔄 Loading weights from /kaggle/working/checkpoints/wav2vec2_vi/best_model.pt...
✅ Model weights loaded successfully.
Evaluating 200 samples...

📊 FINAL TEST SET RESULTS:
   Test Loss: 1.1863
   Test WER:  0.4503
💾 Saved test predictions to /kaggle/working/checkpoints/wav2vec2_vi/test_predictions.json


In [10]:
# Chạy trên Kaggle để dọn dẹp dung lượng đĩa
!rm -f /kaggle/working/checkpoints/wav2vec2_vi/checkpoint_step_*.pt
print("✅ Đã dọn dẹp các checkpoint phụ để giải phóng bộ nhớ!")

✅ Đã dọn dẹp các checkpoint phụ để giải phóng bộ nhớ!


In [11]:
!zip -r /kaggle/working/wav2vec2_vi_best_model.zip \
    /kaggle/working/checkpoints/wav2vec2_vi/best_model.pt \
    /kaggle/working/checkpoints/wav2vec2_vi/best_model_hf

  adding: kaggle/working/checkpoints/wav2vec2_vi/best_model.pt (deflated 8%)
  adding: kaggle/working/checkpoints/wav2vec2_vi/best_model_hf/ (stored 0%)
  adding: kaggle/working/checkpoints/wav2vec2_vi/best_model_hf/vocab.json (deflated 61%)
  adding: kaggle/working/checkpoints/wav2vec2_vi/best_model_hf/config.json (deflated 66%)
  adding: kaggle/working/checkpoints/wav2vec2_vi/best_model_hf/model.safetensors (deflated 9%)
  adding: kaggle/working/checkpoints/wav2vec2_vi/best_model_hf/processor_config.json (deflated 43%)
  adding: kaggle/working/checkpoints/wav2vec2_vi/best_model_hf/tokenizer_config.json (deflated 89%)
  adding: kaggle/working/checkpoints/wav2vec2_vi/best_model_hf/added_tokens.json (deflated 20%)
